In [1]:
import os
import torch
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from transformers import ASTModel, ASTFeatureExtractor
from pathlib import Path

# ── 1. CONFIGURACIÓN Y RUTAS ──
PROJECT_ROOT = Path.cwd().parent
DATASET_TSV = PROJECT_ROOT / 'mtg-jamendo-dataset' / 'data' / 'autotagging_moodtheme.tsv'
AUDIO_DIR = PROJECT_ROOT / 'data' / 'audio' 
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'embeddings_fase3'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 2. CARGAR DATASET (Usamos el Custom Parser fiable de la Fase 2) ──
print("Cargando dataset...")
registros = []
with open(DATASET_TSV, 'r', encoding='utf-8') as f:
    next(f)
    for linea in f:
        if not linea.strip(): continue
        columnas = linea.strip().split('\t')
        if len(columnas) >= 6:
            track_id = columnas[0].replace('track_', '').lstrip('0') 
            if track_id == '': track_id = '0'
            registros.append({
                'track_id': track_id,
                'path': columnas[3]
            })

df_completo = pd.DataFrame(registros)
print(f"Total de canciones en índice: {len(df_completo)}")

# ── 3. CONFIGURACIÓN DEL MODELO AST (Rama Visual) ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Usamos el AST pre-entrenado estándar de HuggingFace
model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = ASTFeatureExtractor.from_pretrained(model_name)
model = ASTModel.from_pretrained(model_name).to(device)
model.eval()

# ── 4. EXTRACCIÓN TEMPORAL (30 SEGUNDOS -> 30 EMBEDDINGS) ──
ast_temporal_embeddings = []
track_ids_validos = []

print("Iniciando extracción de secuencias temporales (30 frames/audio)...")

# Parámetros de la ventana
duracion_total = 30.0
ventana_segundos = 1.0
sr = 16000 # AST requiere estrictamente 16kHz
muestras_por_ventana = int(sr * ventana_segundos)

for idx, row in tqdm(df_completo.iterrows(), total=len(df_completo)):
    track_id = str(row['track_id'])
    
    # Lógica de rutas (Misma que pulimos en la Fase 2 para los .low.mp3)
    path_limpio = str(row['path']).strip() 
    audio_path_normal = AUDIO_DIR / path_limpio
    audio_path_low = AUDIO_DIR / path_limpio.replace('.mp3', '.low.mp3')

    if os.path.exists(audio_path_normal):
        audio_path = audio_path_normal
    elif os.path.exists(audio_path_low):
        audio_path = audio_path_low
    else:
        continue 
        
    try:
        # Cargar exactamente 30 segundos a 16kHz
        audio_array, _ = librosa.load(str(audio_path), sr=sr, duration=duracion_total)
        
        # Si el audio dura menos de 30s (raro en Jamendo, pero por seguridad), rellenamos con ceros (Padding)
        if len(audio_array) < int(sr * duracion_total):
            pad_length = int(sr * duracion_total) - len(audio_array)
            audio_array = np.pad(audio_array, (0, pad_length), mode='constant')
            
        # Trocear el audio en 30 fragmentos de 1 segundo (Shape: 30 x 16000)
        chunks = [audio_array[i:i + muestras_por_ventana] for i in range(0, len(audio_array), muestras_por_ventana)]
        
        # Omitir el último si es más corto por errores de redondeo
        if len(chunks[-1]) < muestras_por_ventana:
            chunks = chunks[:-1]
            
        # Procesar los 30 trozos de golpe en la GPU (Batching) para que vuele
        inputs = feature_extractor(chunks, sampling_rate=sr, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            # AST devuelve un vector 'pooler_output' de 768 dimensiones por cada chunk
            # Resultado: Matriz de (30, 768)
            secuencia_temporal = outputs.pooler_output.cpu().numpy()
            
        ast_temporal_embeddings.append(secuencia_temporal)
        track_ids_validos.append(track_id)
            
    except Exception as e:
        pass # Ignorar audios corruptos

# ── 5. GUARDAR MATRIZ TRIDIMENSIONAL ──
print("\n Guardando resultados temporales...")

# Guardamos como array 3D: (Num_Canciones, 30_Frames, 768_Dimensiones)
np.save(OUTPUT_DIR / 'ast_temporal_embeddings.npy', np.array(ast_temporal_embeddings))
np.save(OUTPUT_DIR / 'track_ids_ast_temporal.npy', np.array(track_ids_validos))

print(f"¡Completado! Se extrajeron las secuencias de {len(track_ids_validos)} canciones.")
print(f"Dimensiones de la matriz final: {np.array(ast_temporal_embeddings).shape}")

c:\Users\Usuario\miniconda3\envs\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando dataset...
Total de canciones en índice: 18486
Usando dispositivo: cuda


c:\Users\Usuario\miniconda3\envs\gpu_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--MIT--ast-finetuned-audioset-10-10-0.4593. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falli

Iniciando extracción de secuencias temporales (30 frames/audio)...


100%|██████████| 18486/18486 [7:57:08<00:00,  1.55s/it]  



 Guardando resultados temporales...
¡Completado! Se extrajeron las secuencias de 18486 canciones.
Dimensiones de la matriz final: (18486, 30, 768)
